# SUCH AN EASY TASK: SARCASM DETECTION IN TEXT...

**Author:** Éloïse Delerue, Lucile Lapray, Léna Rebours

**Date:** April 2026

## 1. Library Import

In [6]:
import pandas as pd
import numpy as np
import csv

## 2. Data Import & Data Manipulation

### 2.1. Data Import

#### 2.1.1. Sarcasm Corpus V2

In [2]:
csv_url = "https://github.com/soraby/sarcasm2/blob/main/sarcasm_v2/GEN-sarc-notsarc.csv?raw=true"

df_SCV2 = pd.read_csv(csv_url)

df_SCV2.head()

,class,id,text
0,notsarc,1,"If that's true, then Freedom of Speech is doom..."
1,notsarc,2,Neener neener - is it time to go in from the p...
2,notsarc,3,"Just like the plastic gun fear, the armour pie..."
3,notsarc,4,So geology is a religion because we weren't he...
4,notsarc,5,Well done Monty. Mark that up as your first ev...


#### 2.1.2. iSarcasmEval

In [3]:
url = "https://raw.githubusercontent.com/iabufarha/iSarcasmEval/main/train/train.En.csv"

df_iS = pd.read_csv(url)

df_iS.head()

,Unnamed: 0,tweet,sarcastic,rephrase,sarcasm,irony,satire,understatement,overstatement,rhetorical_question
0,0,The only thing I got from college is a caffein...,1,"College is really difficult, expensive, tiring...",0.0,1.0,0.0,0.0,0.0,0.0
1,1,I love it when professors draw a big question ...,1,I do not like when professors don’t write out ...,1.0,0.0,0.0,0.0,0.0,0.0
2,2,Remember the hundred emails from companies whe...,1,"I, at the bare minimum, wish companies actuall...",0.0,1.0,0.0,0.0,0.0,0.0
3,3,Today my pop-pop told me I was not “forced” to...,1,"Today my pop-pop told me I was not ""forced"" to...",1.0,0.0,0.0,0.0,0.0,0.0
4,4,@VolphanCarol @littlewhitty @mysticalmanatee I...,1,I would say Ted Cruz is an asshole and doesn’t...,1.0,0.0,0.0,0.0,0.0,0.0


#### 2.1.3. Conversational Sarcasm  Corpus (CSC)

In [7]:
url = "https://raw.githubusercontent.com/CoPsyN/CSC/main/data_full.csv"

df_CSC = pd.read_csv(
    url,
    engine="python",
    sep=",",
    quotechar='"',
    escapechar="\\",
    quoting=csv.QUOTE_MINIMAL,
    on_bad_lines="skip"
)

print(df_CSC.shape)
df_CSC.head()

(31984, 9)


,global_context_name,context_type,context_text,response_text,interlocutor_relationship,global_speaker,global_evaluator,sarcasm_score_by_speaker,sarcasm_score_by_evaluator
0,101,non_neutral,Steve's new year's resolution was to learn Fre...,"Three weeks in and here you are, are you ever ...",best_friend,1001,1001,3,6
1,101,non_neutral,Steve's new year's resolution was to learn Fre...,"Three weeks in and here you are, are you ever ...",best_friend,1001,1002,3,5
2,101,non_neutral,Steve's new year's resolution was to learn Fre...,"Three weeks in and here you are, are you ever ...",best_friend,1001,1003,3,2
3,101,non_neutral,Steve's new year's resolution was to learn Fre...,"Three weeks in and here you are, are you ever ...",best_friend,1001,1004,3,1
4,101,non_neutral,Steve's new year's resolution was to learn Fre...,"Three weeks in and here you are, are you ever ...",best_friend,1001,1005,3,1


### 2.2. Data Manipulation

#### 2.2.1. CSC

In [8]:
# Since there are several annotations per response, we do the mean of the scores given by multiple annotators to the same response / line
df_grouped = df_CSC.groupby(
    ["context_text", "response_text"],
    as_index=False
).agg({
    "sarcasm_score_by_evaluator": "mean"
})
df_grouped

,context_text,response_text,sarcasm_score_by_evaluator
0,"About two years ago, Steve spent half a year i...","And in Thailand they eat monkey arms, fancy a ...",3.750000
1,"About two years ago, Steve spent half a year i...",And what way is that?,4.750000
2,"About two years ago, Steve spent half a year i...",Change the record,3.000000
3,"About two years ago, Steve spent half a year i...","Cool story Steve, change the record",5.250000
4,"About two years ago, Steve spent half a year i...",Cool story bro,5.000000
...,...,...,...
6873,You've just introduced Steve to your work coll...,that's the life everyday,3.500000
6874,You've just introduced Steve to your work coll...,they are a friendly bunch of guys,1.666667
6875,You've just introduced Steve to your work coll...,well this should not last long,3.833333
6876,You've just introduced Steve to your work coll...,yeah i knew everyone would like you,2.000000


In [9]:
# If the mean is above 4, then it is sarcastic (same process as in the paper)
df_grouped["label"] = (df_grouped["sarcasm_score_by_evaluator"] >= 4).astype(int)
df_grouped

,context_text,response_text,sarcasm_score_by_evaluator,label
0,"About two years ago, Steve spent half a year i...","And in Thailand they eat monkey arms, fancy a ...",3.750000,0
1,"About two years ago, Steve spent half a year i...",And what way is that?,4.750000,1
2,"About two years ago, Steve spent half a year i...",Change the record,3.000000,0
3,"About two years ago, Steve spent half a year i...","Cool story Steve, change the record",5.250000,1
4,"About two years ago, Steve spent half a year i...",Cool story bro,5.000000,1
...,...,...,...,...
6873,You've just introduced Steve to your work coll...,that's the life everyday,3.500000,0
6874,You've just introduced Steve to your work coll...,they are a friendly bunch of guys,1.666667,0
6875,You've just introduced Steve to your work coll...,well this should not last long,3.833333,0
6876,You've just introduced Steve to your work coll...,yeah i knew everyone would like you,2.000000,0


In [10]:
csv_url = "https://github.com/soraby/sarcasm2/blob/main/sarcasm_v2/GEN-sarc-notsarc.csv?raw=true"
df_SCV2 = pd.read_csv(csv_url)

# Map 'sarc' to 1 and 'notsarc' to 0 for df_SCV2
df_SCV2['class'] = df_SCV2['class'].map({'sarc': 1, 'notsarc': 0})
df_SCV2.head()

,class,id,text
0,0,1,"If that's true, then Freedom of Speech is doom..."
1,0,2,Neener neener - is it time to go in from the p...
2,0,3,"Just like the plastic gun fear, the armour pie..."
3,0,4,So geology is a religion because we weren't he...
4,0,5,Well done Monty. Mark that up as your first ev...


## 5. Merge datasets

In [11]:
# Standardize each dataset to the same schema
df_csc = pd.DataFrame({
    "text": df_grouped["response_text"],
    "label": df_grouped["label"],
    "source": "CSC"
})

df_is = pd.DataFrame({
    "text": df_iS["tweet"],
    "label": df_iS["sarcastic"],
    "source": "iSarcasmEval"
})

df_scv2 = pd.DataFrame({
    "text": df_SCV2["text"],
    "label": df_SCV2["class"],
    "source": "Sarcasm Corpus V2"
})

# Concatenate datasets
df_combined = pd.concat(
    [df_csc, df_is, df_scv2],
    ignore_index=True
)

# Add row ID column
df_combined.insert(0, "id", range(len(df_combined)))

# Reorder columns
df_combined = df_combined[["id", "text", "label", "source"]]

# Optional: ensure label is binary integer
df_combined["label"] = df_combined["label"].astype(int)

# Save to CSV
df_combined.to_csv("combined_sarcasm_dataset.csv", index=False)

In [ ]:
df_combined

,id,text,label,source
0,0,"And in Thailand they eat monkey arms, fancy a ...",0,CSC
1,1,And what way is that?,1,CSC
2,2,Change the record,0,CSC
3,3,"Cool story Steve, change the record",1,CSC
4,4,Cool story bro,1,CSC
...,...,...,...,...
16861,16861,depends on when the baby bird died. run alon...,1,Sarcasm Corpus V2
16862,16862,"ok, sheesh, to clarify, women who arent aborti...",1,Sarcasm Corpus V2
16863,16863,so.. eh?? hows this sound? will it fly w...,1,Sarcasm Corpus V2
16864,16864,"I think we should put to a vote, the right of ...",1,Sarcasm Corpus V2
